# Project 17 — Local Personal Wiki Copilot

## Query Exported Notes and Wiki Files Locally

**Goal:** Build a local personal-knowledge assistant that ingests markdown
wiki pages with folder and tag metadata, indexes them for semantic search,
answers questions across the entire note corpus, navigates note links,
and produces a related-notes graph — all running locally with Ollama.

**Stack:** Ollama · LlamaIndex · Jupyter

```
Markdown wiki pages ──► Documents (with folder / tag metadata)
                                    │
                          Ollama Embeddings ──► VectorStoreIndex
                                                    │
User question ──► retrieval ──► relevant pages ──► LLM ──► grounded answer
                                                             │
                                     source pages + links ──► wiki navigation
```

### What You'll Learn

1. Index personal wiki pages preserving folder and tag metadata
2. Cross-page semantic search across a note corpus
3. Metadata-filtered retrieval (by folder, tag, or page type)
4. Note-link-aware answers that cite source pages
5. Conversational wiki chat with follow-up context
6. Related-notes discovery via lightweight graph traversal

### Prerequisites

- **Ollama** running locally with `nomic-embed-text` and `qwen3:8b`
- Python 3.9+

In [ ]:
# Install dependencies (uncomment and run once)
!pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama

## Step 1 — Verify Ollama Is Running

Before using any local model, we confirm that Ollama is available at
`localhost:11434` and list downloaded models.

In [ ]:
import requests

OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama is running — {len(models)} model(s) available")
    print("   Models:", ", ".join(models[:10]))
except Exception as e:
    print(f"❌ Cannot reach Ollama at {OLLAMA_BASE}: {e}")
    print("   Start Ollama first:  ollama serve")

## Step 2 — Configure LLM and Embeddings

We use LlamaIndex's `Settings` to configure the local LLM and embedding
model globally. Every index and query engine will inherit these defaults.

In [ ]:
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

LLM_MODEL = "qwen3:8b"
EMBED_MODEL = "nomic-embed-text"

Settings.llm = Ollama(model=LLM_MODEL, request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name=EMBED_MODEL)

# Quick smoke test
vec = Settings.embed_model.get_query_embedding("test")
print(f"✅ Embedding model ready — dimension: {len(vec)}")
resp = Settings.llm.complete("Say hello in one sentence.")
print(f"✅ LLM ready — response: {str(resp)[:100]}")

## Step 3 — Create Sample Personal Wiki

A real personal wiki contains many interconnected notes across
folders like `projects/`, `recipes/`, `learning/`, `meetings/`.

We simulate a realistic personal wiki with **10 pages** spanning
different categories, each with:
- `page_title` — the note title
- `folder` — the wiki folder it lives in
- `tags` — comma-separated tags for filtering
- `links_to` — other pages this note references (wiki links)

This metadata lets us do filtered retrieval and link-aware answers.

In [ ]:
from llama_index.core import Document

wiki_pages = [
    # ── Projects ──────────────────────────────────────────────
    Document(
        text=(
            "# Project Phoenix\n"
            "Status: Active | Lead: Alice | Started: 2024-Q3\n\n"
            "## Goals\n"
            "- Migrate legacy monolith to microservices\n"
            "- Reduce deployment time from 4 hours to 15 minutes\n"
            "- Achieve 99.95% uptime SLA\n\n"
            "## Architecture Decisions\n"
            "- ADR-001: Use Kubernetes for orchestration (approved)\n"
            "- ADR-002: Use PostgreSQL over MySQL (approved)\n"
            "- ADR-003: Event-driven architecture with Kafka (in review)\n\n"
            "## Current Status\n"
            "Phase 1 (Auth service) complete. Phase 2 (Payment service) in progress.\n"
            "Blocked by: Legacy database schema migration.\n\n"
            "## Related\n"
            "See also: [[Dev Setup Guide]], [[Incident Response Playbook]]"
        ),
        metadata={"page_title": "Project Phoenix", "folder": "projects",
                  "tags": "architecture,migration,kubernetes",
                  "links_to": "Dev Setup Guide,Incident Response Playbook"}),

    Document(
        text=(
            "# Home Automation Project\n"
            "Status: Planning | Started: 2025-Q1\n\n"
            "## Goals\n"
            "- Set up Home Assistant on Raspberry Pi 5\n"
            "- Automate lighting, thermostat, and security cameras\n"
            "- Create voice-activated routines via local speech model\n\n"
            "## Hardware\n"
            "- Raspberry Pi 5 (8 GB) — ordered\n"
            "- Zigbee USB coordinator (SONOFF Zigbee 3.0)\n"
            "- 4x Philips Hue bulbs, 2x Aqara door sensors\n\n"
            "## Software Stack\n"
            "Home Assistant OS + Zigbee2MQTT + Node-RED for automations.\n"
            "Local Whisper for voice commands (no cloud dependency).\n\n"
            "## Related\n"
            "See also: [[Raspberry Pi Setup Notes]], [[Shopping List]]"
        ),
        metadata={"page_title": "Home Automation Project", "folder": "projects",
                  "tags": "iot,home-assistant,raspberry-pi",
                  "links_to": "Raspberry Pi Setup Notes,Shopping List"}),

    # ── Work / Processes ──────────────────────────────────────
    Document(
        text=(
            "# Onboarding Checklist\n\n"
            "## Week 1\n"
            "- [ ] Set up dev environment (see [[Dev Setup Guide]])\n"
            "- [ ] Complete security training\n"
            "- [ ] Get access to GitHub, Jira, Slack\n"
            "- [ ] Meet with team lead and buddy\n\n"
            "## Week 2\n"
            "- [ ] Complete first PR (good-first-issue label)\n"
            "- [ ] Shadow a production deployment\n"
            "- [ ] Read Architecture Decision Records\n\n"
            "## Tools\n"
            "- IDE: VS Code with recommended extensions\n"
            "- Terminal: Use Oh My Zsh with our custom theme\n"
            "- VPN: Required for staging/production access"
        ),
        metadata={"page_title": "Onboarding Checklist", "folder": "work",
                  "tags": "hr,new-hire,process",
                  "links_to": "Dev Setup Guide"}),

    Document(
        text=(
            "# Incident Response Playbook\n\n"
            "## Severity Levels\n"
            "- SEV1: Customer-facing outage, all hands on deck\n"
            "- SEV2: Degraded service, team lead + on-call\n"
            "- SEV3: Internal tooling issue, on-call engineer\n\n"
            "## Response Steps\n"
            "1. Acknowledge the alert within 5 minutes\n"
            "2. Create incident channel: #inc-YYYY-MM-DD-brief-description\n"
            "3. Assign Incident Commander and Communications Lead\n"
            "4. Begin investigation, post updates every 15 min for SEV1\n"
            "5. After resolution: blameless postmortem within 48 hours\n\n"
            "## Escalation Path\n"
            "On-call → Team Lead → Engineering Director → CTO\n\n"
            "## Related\n"
            "See also: [[Project Phoenix]], [[Monitoring Dashboard Notes]]"
        ),
        metadata={"page_title": "Incident Response Playbook", "folder": "work",
                  "tags": "ops,incidents,runbook",
                  "links_to": "Project Phoenix,Monitoring Dashboard Notes"}),

    Document(
        text=(
            "# Dev Setup Guide\n\n"
            "## Prerequisites\n"
            "- macOS 14+ or Ubuntu 22.04+\n"
            "- Docker Desktop or Rancher Desktop\n"
            "- Node.js 20 LTS, Python 3.11+, Go 1.22+\n\n"
            "## Getting Started\n"
            "1. Clone the monorepo: `git clone git@github.com:acme/platform.git`\n"
            "2. Run `make bootstrap` — installs all dependencies\n"
            "3. Copy `.env.example` to `.env` and fill in secrets from Vault\n"
            "4. Start local stack: `docker compose up -d`\n"
            "5. Verify: `make test-smoke`\n\n"
            "## Common Issues\n"
            "- Docker out of disk: `docker system prune -a`\n"
            "- Port conflict on 5432: Check if another PostgreSQL is running\n"
            "- M1/M2 ARM issues: Use `--platform linux/amd64` for legacy images\n\n"
            "## Related\n"
            "See also: [[Onboarding Checklist]], [[Project Phoenix]]"
        ),
        metadata={"page_title": "Dev Setup Guide", "folder": "work",
                  "tags": "dev,setup,docker",
                  "links_to": "Onboarding Checklist,Project Phoenix"}),

    # ── Learning ──────────────────────────────────────────────
    Document(
        text=(
            "# Rust Learning Notes\n\n"
            "## Key Concepts\n"
            "- Ownership: Each value has exactly one owner. When the owner goes\n"
            "  out of scope, the value is dropped.\n"
            "- Borrowing: References (&T) let you read without taking ownership.\n"
            "  Mutable references (&mut T) allow modification (only one at a time).\n"
            "- Lifetimes: Compiler tracks how long references are valid to prevent\n"
            "  dangling pointers at compile time.\n\n"
            "## Useful Patterns\n"
            "- Use `match` instead of if-else chains for enums\n"
            "- `Option<T>` replaces null; `Result<T, E>` replaces exceptions\n"
            "- The `?` operator propagates errors cleanly\n\n"
            "## Resources\n"
            "- The Rust Book (official): doc.rust-lang.org/book\n"
            "- Rustlings exercises: github.com/rust-lang/rustlings\n"
            "- Jon Gjengset's YouTube channel for advanced topics"
        ),
        metadata={"page_title": "Rust Learning Notes", "folder": "learning",
                  "tags": "rust,programming,language",
                  "links_to": ""}),

    Document(
        text=(
            "# Machine Learning Study Plan\n\n"
            "## Phase 1 — Foundations (Months 1-2)\n"
            "- Linear algebra: vectors, matrices, eigenvalues\n"
            "- Probability: Bayes theorem, distributions, MLE\n"
            "- Python + NumPy + pandas fluency\n\n"
            "## Phase 2 — Classical ML (Months 3-4)\n"
            "- Supervised: linear/logistic regression, decision trees, SVM, k-NN\n"
            "- Unsupervised: k-means, PCA, DBSCAN\n"
            "- Evaluation: cross-validation, ROC-AUC, confusion matrix\n\n"
            "## Phase 3 — Deep Learning (Months 5-6)\n"
            "- Neural networks, backpropagation, CNNs, RNNs/LSTMs\n"
            "- Transformers and attention mechanism\n"
            "- Practical: PyTorch, Hugging Face Transformers\n\n"
            "## Resources\n"
            "- Andrew Ng's Coursera specialization\n"
            "- Fast.ai Practical Deep Learning course\n"
            "- Hands-On ML (Aurélien Géron)\n\n"
            "## Related\n"
            "See also: [[Rust Learning Notes]]"
        ),
        metadata={"page_title": "Machine Learning Study Plan", "folder": "learning",
                  "tags": "ml,study,deep-learning",
                  "links_to": "Rust Learning Notes"}),

    # ── Personal ──────────────────────────────────────────────
    Document(
        text=(
            "# Sourdough Bread Recipe\n\n"
            "## Ingredients\n"
            "- 500g bread flour (high protein)\n"
            "- 350g water (70% hydration)\n"
            "- 100g active sourdough starter\n"
            "- 10g salt\n\n"
            "## Method\n"
            "1. Autolyse: Mix flour + water, rest 30 min\n"
            "2. Add starter and salt, fold to incorporate\n"
            "3. Bulk ferment 4-5 hours at 24°C (stretch & fold every 30 min x4)\n"
            "4. Pre-shape, bench rest 20 min, final shape\n"
            "5. Cold retard overnight in fridge (12-16 hours)\n"
            "6. Bake in Dutch oven: 250°C lid on 20 min, lid off 20 min\n\n"
            "## Tips\n"
            "- Starter should double in 4-6 hours before use\n"
            "- Higher hydration = more open crumb but harder to handle\n"
            "- Score deeply with a lame just before baking"
        ),
        metadata={"page_title": "Sourdough Bread Recipe", "folder": "recipes",
                  "tags": "baking,sourdough,bread",
                  "links_to": ""}),

    Document(
        text=(
            "# Travel — Japan 2025 Plan\n\n"
            "## Itinerary\n"
            "- Day 1-3: Tokyo (Shibuya, Akihabara, Tsukiji Outer Market)\n"
            "- Day 4-5: Hakone (onsen, Mt. Fuji views)\n"
            "- Day 6-8: Kyoto (Fushimi Inari, Arashiyama bamboo, tea ceremony)\n"
            "- Day 9-10: Osaka (street food, Dotonbori, day trip to Nara)\n\n"
            "## Budget\n"
            "- Flights: ~$900 round trip\n"
            "- Rail pass: $280 (14-day JR Pass)\n"
            "- Accommodation: ~$100/night (mix of hotel + ryokan)\n"
            "- Food: ~$50/day\n"
            "- Total estimate: ~$3,200\n\n"
            "## Notes\n"
            "- Book ryokan in Hakone early — sells out fast\n"
            "- Get a Suica card for local transport\n"
            "- Pocket WiFi > SIM for convenience"
        ),
        metadata={"page_title": "Travel Japan 2025", "folder": "personal",
                  "tags": "travel,japan,planning",
                  "links_to": ""}),

    Document(
        text=(
            "# Monitoring Dashboard Notes\n\n"
            "## Current Setup\n"
            "- Grafana for dashboards (metrics + logs)\n"
            "- Prometheus for metrics collection (15s scrape interval)\n"
            "- Loki for log aggregation\n"
            "- AlertManager for routing alerts to PagerDuty + Slack\n\n"
            "## Key Dashboards\n"
            "- Service Health: p50/p95/p99 latency, error rate, throughput\n"
            "- Infrastructure: CPU, memory, disk I/O per node\n"
            "- Business Metrics: signups, orders, payment success rate\n\n"
            "## Alert Rules\n"
            "- Error rate > 1%: SEV3 alert\n"
            "- Error rate > 5%: SEV2 alert\n"
            "- p99 latency > 2s: SEV3 alert\n"
            "- Service down > 1 min: SEV1 alert\n\n"
            "## Related\n"
            "See also: [[Incident Response Playbook]], [[Project Phoenix]]"
        ),
        metadata={"page_title": "Monitoring Dashboard Notes", "folder": "work",
                  "tags": "monitoring,grafana,alerting",
                  "links_to": "Incident Response Playbook,Project Phoenix"}),
]

print(f"✅ Created {len(wiki_pages)} wiki pages")
print()
print("Pages by folder:")
from collections import Counter
folder_counts = Counter(d.metadata["folder"] for d in wiki_pages)
for folder, count in sorted(folder_counts.items()):
    print(f"  📁 {folder}: {count} pages")

## Step 4 — Build the Wiki Search Index

We use LlamaIndex `VectorStoreIndex` to embed and index all wiki pages.
Each page becomes a searchable document with its metadata preserved,
so retrieval results include the page title, folder, and tags.

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(wiki_pages, show_progress=True)
print(f"✅ Wiki index built — {len(wiki_pages)} pages indexed")

## Step 5 — Basic Wiki Search (Cross-Page Retrieval)

The simplest use case: ask a question and get an answer synthesized
from the most relevant wiki pages. We configure the query engine to
retrieve the top 3 most similar pages.

**Why this matters:** A personal wiki can have hundreds of pages. Semantic
search finds relevant information even when the user doesn't remember
the exact page title or wording.

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=3)

queries = [
    "What's the current status of Project Phoenix?",
    "What do I need to do in my first week as a new hire?",
    "What's the process for a SEV1 incident?",
    "What database did we choose and why?",
    "How do I bake sourdough bread?",
]

print("=" * 70)
print("WIKI SEARCH RESULTS")
print("=" * 70)

for q in queries:
    print(f"\n❓ Q: {q}")
    response = query_engine.query(q)
    print(f"💡 A: {response}")
    pages = [n.metadata.get("page_title", "?") for n in response.source_nodes]
    print(f"📄 Source pages: {pages}")
    print("-" * 70)

## Step 6 — Metadata-Filtered Retrieval

Personal wikis often need filtered queries like:
- "Search only in my work notes"
- "Find pages tagged with 'architecture'"

LlamaIndex supports metadata filtering at retrieval time via
`MetadataFilters`. This narrows the search space before computing
similarity, giving more precise results.

In [ ]:
from llama_index.core.vector_stores import (
    MetadataFilter,
    MetadataFilters,
    FilterOperator,
)

# --- Filter by folder: only search 'work' pages ---
work_filters = MetadataFilters(
    filters=[MetadataFilter(key="folder", value="work", operator=FilterOperator.EQ)]
)
work_engine = index.as_query_engine(
    similarity_top_k=3,
    filters=work_filters,
)

print("=" * 70)
print("FILTERED SEARCH — Work folder only")
print("=" * 70)

q = "What monitoring tools do we use?"
print(f"\n❓ Q: {q}")
resp = work_engine.query(q)
print(f"💡 A: {resp}")
print(f"📄 Source pages: {[n.metadata.get('page_title', '?') for n in resp.source_nodes]}")

# --- Filter by folder: only search 'learning' pages ---
learn_filters = MetadataFilters(
    filters=[MetadataFilter(key="folder", value="learning", operator=FilterOperator.EQ)]
)
learn_engine = index.as_query_engine(
    similarity_top_k=3,
    filters=learn_filters,
)

print()
print("=" * 70)
print("FILTERED SEARCH — Learning folder only")
print("=" * 70)

q2 = "What should I learn about neural networks?"
print(f"\n❓ Q: {q2}")
resp2 = learn_engine.query(q2)
print(f"💡 A: {resp2}")
print(f"📄 Source pages: {[n.metadata.get('page_title', '?') for n in resp2.source_nodes]}")

## Step 7 — Custom Wiki-Aware Prompt

We override the default QA prompt to make the assistant behave like
a wiki copilot: it cites page titles, mentions related pages via
wiki links, and admits when it can't find relevant information.

In [ ]:
from llama_index.core import PromptTemplate

WIKI_QA_PROMPT = PromptTemplate(
    "You are a helpful personal wiki copilot. You answer questions using\n"
    "only the wiki pages provided below.\n\n"
    "Rules:\n"
    "1. Always cite which wiki page(s) your answer comes from.\n"
    "2. If a page links to other pages (via [[Page Name]]), mention them.\n"
    "3. If no relevant wiki page exists, say so honestly.\n"
    "4. Keep answers concise and well-structured.\n\n"
    "Wiki pages:\n"
    "{context_str}\n\n"
    "Question: {query_str}\n\n"
    "Answer:"
)

wiki_engine = index.as_query_engine(
    similarity_top_k=3,
    text_qa_template=WIKI_QA_PROMPT,
)

print("=" * 70)
print("WIKI COPILOT — Citation-Aware Answers")
print("=" * 70)

wiki_queries = [
    "How do I set up the dev environment? What other pages should I read?",
    "What alert rules trigger a SEV1 incident?",
    "What is Rust's ownership model?",
]

for q in wiki_queries:
    print(f"\n❓ Q: {q}")
    resp = wiki_engine.query(q)
    print(f"💡 A: {resp}")
    print("-" * 70)

## Step 8 — Conversational Wiki Chat with Follow-Ups

A wiki copilot should handle multi-turn conversations:
1. User asks about a topic
2. Follow-up drills deeper
3. Context from previous turns is preserved

We use LlamaIndex's `CondenseQuestionChatEngine` which reformulates
follow-up questions by incorporating chat history before retrieval.

In [ ]:
from llama_index.core.chat_engine import CondenseQuestionChatEngine

chat_engine = CondenseQuestionChatEngine.from_defaults(
    query_engine=wiki_engine,
)

# Simulate a multi-turn conversation
conversation = [
    "Tell me about our architecture decisions",
    "Which one is still in review?",
    "What event system are we considering and what is the current project status?",
]

print("=" * 70)
print("CONVERSATIONAL WIKI CHAT")
print("=" * 70)

for msg in conversation:
    print(f"\n👤 User: {msg}")
    resp = chat_engine.chat(msg)
    print(f"🤖 Wiki: {resp}")
    print("-" * 70)

# Reset for next conversation
chat_engine.reset()
print("\n(Chat history reset)")

## Step 9 — Note Link Graph Discovery

One thing that makes a personal wiki special is **links between notes**.
When you read a page, you want to discover related pages — not just
by semantic similarity, but by explicit `[[wiki links]]`.

We build a lightweight note-link graph from the `links_to` metadata
and provide a function that, given a page, shows:
1. Pages it links to (outgoing)
2. Pages that link back to it (incoming / backlinks)

This mimics tools like Obsidian's graph view or Notion's backlinks.

In [ ]:
from collections import defaultdict

# Build adjacency list from wiki link metadata
outgoing_links = {}  # page -> list of pages it links to
incoming_links = defaultdict(list)  # page -> list of pages linking to it

for doc in wiki_pages:
    title = doc.metadata["page_title"]
    links_str = doc.metadata.get("links_to", "")
    targets = [l.strip() for l in links_str.split(",") if l.strip()]
    outgoing_links[title] = targets
    for target in targets:
        incoming_links[target].append(title)


def show_note_graph(page_title: str) -> None:
    """Display outgoing and incoming links for a wiki page."""
    print(f"\n📄 {page_title}")
    out = outgoing_links.get(page_title, [])
    inc = incoming_links.get(page_title, [])
    print(f"  ➡️  Links to:       {out if out else '(none)'}")
    print(f"  ⬅️  Backlinks from: {inc if inc else '(none)'}")


print("=" * 70)
print("NOTE LINK GRAPH")
print("=" * 70)

# Show graph for key pages
show_note_graph("Project Phoenix")
show_note_graph("Dev Setup Guide")
show_note_graph("Incident Response Playbook")
show_note_graph("Machine Learning Study Plan")
show_note_graph("Sourdough Bread Recipe")

## Step 10 — Related Notes Recommendation

Combining semantic similarity with wiki links gives better
"related notes" recommendations. Given a page:
1. Find semantically similar pages via embedding search
2. Add explicitly linked pages from the link graph
3. Merge and rank the combined set

This approach surfaces connections that pure keyword or pure
embedding search alone would miss.

In [ ]:
retriever = index.as_retriever(similarity_top_k=5)


def find_related_notes(page_title: str) -> None:
    """Find related notes via semantic search + link graph."""
    print(f"\n🔍 Related notes for: '{page_title}'")
    print("-" * 50)

    # 1. Semantic neighbours: query with the page title
    nodes = retriever.retrieve(page_title)
    semantic_matches = []
    for node in nodes:
        title = node.metadata.get("page_title", "?")
        if title != page_title:  # exclude self
            semantic_matches.append((title, round(node.score, 3)))

    print("  📊 By semantic similarity:")
    if semantic_matches:
        for title, score in semantic_matches[:4]:
            print(f"     • {title} (score: {score})")
    else:
        print("     (none found)")

    # 2. Explicit wiki links
    out = outgoing_links.get(page_title, [])
    inc = incoming_links.get(page_title, [])
    linked = set(out + inc) - {page_title}
    print(f"  🔗 By wiki links: {sorted(linked) if linked else '(none)'}")

    # 3. Merged recommendations
    all_related = set(t for t, _ in semantic_matches) | linked
    print(f"  ✅ Combined related: {sorted(all_related)}")


find_related_notes("Project Phoenix")
find_related_notes("Incident Response Playbook")
find_related_notes("Machine Learning Study Plan")

## Step 11 — Wiki Summary Generator

A useful wiki copilot feature: generate summaries of entire folders.
This helps users get a quick overview of all their notes in a category.

We gather pages by folder and ask the LLM to produce a structured summary.

In [ ]:
def summarize_folder(folder_name: str) -> None:
    """Generate a summary of all wiki pages in a given folder."""
    folder_pages = [d for d in wiki_pages if d.metadata["folder"] == folder_name]
    if not folder_pages:
        print(f"No pages found in folder: {folder_name}")
        return

    titles = [d.metadata["page_title"] for d in folder_pages]
    combined = "\n\n---\n\n".join(
        f"Page: {d.metadata['page_title']}\n{d.text}" for d in folder_pages
    )

    prompt = (
        f"You are summarizing the '{folder_name}' folder of a personal wiki.\n"
        f"It contains {len(folder_pages)} pages: {', '.join(titles)}.\n\n"
        f"Content:\n{combined}\n\n"
        "Produce a concise folder summary with:\n"
        "1. One-sentence overview of the folder\n"
        "2. Key highlights from each page (1-2 bullets each)\n"
        "3. Action items or open questions across pages\n"
    )

    resp = Settings.llm.complete(prompt)
    print(f"\n📁 Folder Summary: {folder_name}")
    print("=" * 60)
    print(str(resp))
    print("=" * 60)


summarize_folder("work")
summarize_folder("learning")

## Step 12 — Tag-Based Page Discovery

Users often tag wiki pages for easy categorization. We build a simple
tag index and let users browse pages by tag — a complement to semantic
search that works well for known categories.

In [ ]:
# Build tag → pages index
tag_index = defaultdict(list)
for doc in wiki_pages:
    tags = [t.strip() for t in doc.metadata.get("tags", "").split(",") if t.strip()]
    for tag in tags:
        tag_index[tag].append(doc.metadata["page_title"])

print("📑 Tag Index")
print("=" * 50)
for tag in sorted(tag_index.keys()):
    pages = tag_index[tag]
    print(f"  #{tag}: {pages}")

print(f"\nTotal tags: {len(tag_index)}")
print(f"Total pages: {len(wiki_pages)}")

## Step 13 — End-to-End Wiki Copilot Demo

Let's put it all together with a realistic scenario: a user exploring
their wiki with mixed queries spanning work, learning, and personal notes.

The copilot should:
- Answer from the right pages
- Handle cross-domain queries
- Cite sources properly
- Handle questions with no relevant wiki content gracefully

In [ ]:
demo_queries = [
    # Cross-domain: mixes work + personal
    "What trips am I planning and what's the budget?",
    # Technical depth
    "Explain the difference between Rust ownership and borrowing",
    # Work process
    "If our error rate exceeds 5%, what happens?",
    # No relevant page
    "What is the company's revenue for last quarter?",
    # Linking across pages
    "What pages mention Kubernetes?",
]

print("=" * 70)
print("END-TO-END WIKI COPILOT DEMO")
print("=" * 70)

for q in demo_queries:
    print(f"\n👤 User: {q}")
    resp = wiki_engine.query(q)
    print(f"🤖 Wiki: {resp}")
    sources = [n.metadata.get("page_title", "?") for n in resp.source_nodes]
    folders = [n.metadata.get("folder", "?") for n in resp.source_nodes]
    print(f"📄 Sources: {sources} (folders: {folders})")
    print("-" * 70)

## Step 14 — Retrieval Quality Inspection

Good RAG practice: inspect what the retriever actually returns before
the LLM synthesizes an answer. This helps debug bad answers caused by
retrieval misses rather than LLM hallucination.

In [ ]:
test_query = "How do I deploy code?"
print(f"🔍 Retrieval debug for: '{test_query}'")
print("=" * 60)

nodes = retriever.retrieve(test_query)
for i, node in enumerate(nodes, 1):
    print(f"\n--- Result #{i} ---")
    print(f"  Page:   {node.metadata.get('page_title', '?')}")
    print(f"  Folder: {node.metadata.get('folder', '?')}")
    print(f"  Tags:   {node.metadata.get('tags', '?')}")
    print(f"  Score:  {node.score:.4f}")
    print(f"  Text:   {node.text[:150]}...")

## Limitations and Tradeoffs

1. **Small demo corpus:** 10 pages is enough to learn the pattern, but
   real wikis have hundreds to thousands of pages. Larger corpora need
   chunking strategies and persistent vector stores.

2. **Flat metadata filtering:** We used simple equality filters.
   Production systems may need full-text tag search, date ranges,
   or hierarchical folder filtering.

3. **Wiki link parsing:** We stored links as comma-separated metadata.
   A real implementation would parse `[[...]]` syntax from markdown
   and resolve aliases automatically.

4. **No persistent storage:** The index is rebuilt every run. For
   production, persist to ChromaDB, Qdrant, or FAISS on disk.

5. **Local LLM quality:** Answer quality depends on the local model.
   Larger models (14B+) give better synthesis but are slower.

6. **No incremental updates:** Adding new pages requires a full
   re-index. A real wiki copilot would support incremental insertion.

## How to Extend This Project

1. **Real wiki import:** Parse actual Obsidian, Logseq, or Notion
   exports using `SimpleDirectoryReader` with metadata extractors.

2. **Persistent index:** Use `StorageContext` with a persistent
   vector store (ChromaDB) to avoid re-indexing every session.

3. **Hybrid search:** Combine BM25 keyword search with embeddings
   for better recall on exact terms (page titles, abbreviations).

4. **Graph index:** Use LlamaIndex's `KnowledgeGraphIndex` to build
   a proper entity-relation graph from wiki content.

5. **Auto-tagging:** Use the LLM to suggest tags for new/untagged pages.

6. **Write-back:** Let the copilot draft new wiki pages or update
   existing ones based on conversation context.

## What You Learned

| Concept | What We Did |
|---|---|
| **Wiki page indexing** | Embedded markdown pages with folder/tag metadata |
| **Cross-page search** | Semantic retrieval across the full note corpus |
| **Metadata filtering** | Filtered search by folder or tag |
| **Custom QA prompt** | Citation-aware wiki copilot prompt |
| **Conversational chat** | Multi-turn follow-ups with context memory |
| **Link graph** | Outgoing links + backlinks discovery |
| **Related notes** | Combined semantic + link-based recommendations |
| **Folder summaries** | LLM-generated overviews of note categories |
| **Tag browsing** | Tag-to-page index for structured navigation |
| **Retrieval debugging** | Inspecting raw retrieval results for quality |